# Lab09 - Announcement JSON Extraction

Qwen3-1.7B를 사용해 공지문에서 `when`, `where`, `items`, `task`, `teacher`를 JSON으로 추출하고, naive/improved prompting 결과를 비교합니다.

**Colab 설정:** `런타임 > 런타임 유형 변경 > T4 GPU`를 선택한 뒤 위에서부터 셀을 실행하세요.

In [ ]:
import torch

assert torch.cuda.is_available(), (
    "GPU를 사용할 수 없습니다. 런타임 > 런타임 유형 변경에서 T4 GPU를 선택하세요."
)
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"PyTorch: {torch.__version__}")

## 1. Google Drive 마운트 및 프로젝트 준비

Google Drive의 `Lab09` 폴더를 Colab 로컬 런타임으로 복사합니다. Drive 경로가 다르면 `DRIVE_LAB_DIR`만 수정하세요.

In [ ]:
from google.colab import drive
from pathlib import Path
import os
import shutil

drive.mount("/content/drive")

DRIVE_LAB_DIR = Path("/content/drive/MyDrive/Lab09")
LOCAL_LAB_DIR = Path("/content/Lab09")

assert DRIVE_LAB_DIR.exists(), (
    f"{DRIVE_LAB_DIR}가 없습니다. Google Drive의 Lab09 경로를 확인하세요."
)
if LOCAL_LAB_DIR.exists():
    shutil.rmtree(LOCAL_LAB_DIR)
shutil.copytree(
    DRIVE_LAB_DIR,
    LOCAL_LAB_DIR,
    ignore=shutil.ignore_patterns("__pycache__", "*.pyc", "results"),
)

os.chdir(LOCAL_LAB_DIR)
print(f"Working directory: {Path.cwd()}")

## 2. 패키지 설치

Colab에 포함된 PyTorch는 그대로 사용하고, Qwen3 실행에 필요한 패키지만 설치합니다.

In [ ]:
%pip install -q -U "transformers>=4.51.0" "accelerate>=1.0.0" "tqdm>=4.66"

## 3. 데이터 확인

In [ ]:
import json
from pathlib import Path

data_path = Path("data/announcements.json")
records = json.loads(data_path.read_text(encoding="utf-8"))

print(f"Number of announcements: {len(records)}")
print(json.dumps(records[0], ensure_ascii=False, indent=2))

## 4. 단일 샘플 실행

첫 실행에서는 Hugging Face에서 약 3~4GB 모델을 다운로드합니다.

In [ ]:
from util.model_runner import ModelRunner
import naive
import improved

runner = ModelRunner()
sample = records[0]

print("[announcement]")
print(sample["text"])
print("\n[gold labels]")
print(json.dumps(sample["labels"], ensure_ascii=False, indent=2))
print("\n[naive output]")
print(naive.run(sample["text"], runner))
print("\n[improved output]")
print(improved.run(sample["text"], runner))

## 5. 평가 실행

`LIMIT = 10`은 빠른 확인용입니다. 전체 100개를 평가하려면 `LIMIT = None`으로 변경하세요. `METHOD`는 `"naive"`, `"improved"`, `"both"` 중 하나입니다.

In [ ]:
from main import STRATEGIES, load_data, run_strategy
from util.evaluate import markdown_table, print_comparison, print_report

LIMIT = 10
METHOD = "both"
MAX_NEW_TOKENS = 256

assert METHOD in {"naive", "improved", "both"}
eval_records = load_data(LIMIT)
methods = ["naive", "improved"] if METHOD == "both" else [METHOD]

results = {}
raw_rows = {}
for name in methods:
    print(f"\n----- running '{name}' -----")
    rows, agg = run_strategy(
        name, STRATEGIES[name], runner, eval_records, MAX_NEW_TOKENS
    )
    raw_rows[name] = rows
    results[name] = agg
    print_report(name, agg)

if len(results) > 1:
    print_comparison(results)

print("\n[README용 Markdown]")
print(markdown_table(results))

## 6. 결과 저장 및 다운로드

In [ ]:
from google.colab import files

drive_results_dir = DRIVE_LAB_DIR / "results"
drive_results_dir.mkdir(exist_ok=True)
output_path = drive_results_dir / "lab09_results.json"
output_path.write_text(
    json.dumps(
        {"metrics": results, "predictions": raw_rows},
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)
print(f"Saved: {output_path.resolve()}")
files.download(str(output_path))